# CIS 536/736 — Week 13 Lab: Unity VFX Graph 17 & Multi-DCC Integration (Fire, Smoke, and Waterfall)

**Course:** CIS 536/736 — Computer Graphics (Fall 2026)  
**Phase 5 / Module 11:** Data/Info Visualization II, VFX & Interim Integration  
**Lecture Anchors:** Lecture 35 (*Special Topics: VFX & Particle Systems Survey*) & Lecture 36 (*Houdini, Blender 4 & Unity 6 Pipeline Synthesis*)  
**Target Runtime:** Unity 6 LTS (`com.unity.visualeffectgraph` **17.x**, URP 17 / HDRP 17), Blender 4.2 LTS, and optional SideFX Houdini (Apprentice/Core + SideFX Labs)

---

## Lab Objectives
This lab bridges Lecture 35's particle/fluid theory and Lecture 36's multi-DCC interchange pipeline by building the **simplest possible, working GPU VFX Graph 17 systems** for three canonical phenomena—each isolated in its own self-contained block:
1. **Block 1 — Fire (Lagrangian Buoyancy + Curl Turbulence + Additive Lifetime Ramp):** Exercises Lecture 35 Slides 2, 5, and 7 (`Initialize` / `Update` / `Output` contexts, Structure-of-Arrays state, upward buoyancy, turbulence, and $t \in [0, 1]$ emission color mapping), paired with a Blender 4.2 emitter export script and a Unity `.pcache` Point Cache.
2. **Block 2 — Smoke Plume (Eulerian Divergence-Free Vector Field + Lit Alpha Billboards):** Exercises Lecture 35 Slides 3 and 6 (*GPU Gems* Ch. 38 incompressibility $\nabla \cdot \mathbf{u} = 0$, billboard overdraw, Soft Particles) and Lecture 36's Houdini-to-Unity 3D Vector Field (`.vf`) workflow by synthesizing and exporting a binary `VF_V` 3D Vector Field directly loadable as a `Texture3D` in Unity VFX Graph 17.
3. **Block 3 — Waterfall (Multi-DCC Coordinate Transform + Gravity + Analytical SDF Plane Collision):** Exercises Lecture 35 Slides 2, 4, and 7 (semi-implicit Euler integration, velocity-stretched quads, SDF collision) and Lecture 36's `Houdini (Y-up) <-> Blender 4.2 (Z-up) -> Unity 6 (LH Y-up)` coordinate alignment by exporting a transformed cliff-lip `.pcache` and simulating falling sheets + basin splash mist.
4. **Block 4 — Unity 6 C# Profiling Harness & Healy Ch. 5–6 Benchmark Visualization:** Exercises Lecture 35 Slide 8 (fill-rate/overdraw & particle budgets) and Lecture 36 Slides 6–7 (mean + 95% CI faceted profiling plots) to prepare your Phase 3 capstone evaluation.

## Setup: Multi-DCC Coordinate Transforms, SoA Helpers & Unity VFX 17 Binary/ASCII Exporters

Per **Lecture 35 (Slide 2)**, GPU particle engines store particle attributes in **Structure-of-Arrays (SoA)** layout rather than Array-of-Structures (AoS) for coalesced memory access. Per **Lecture 36 (Slides 3–5)**, this cell sets up:
- Explicit coordinate transforms from **Blender 4.2 (Right-Handed Z-up)** and **Houdini (Right-Handed Y-up)** into **Unity 6 (Left-Handed Y-up)**.
- A native **Unity VFX Graph 17 ASCII Point Cache (`.pcache`)** writer (`position`, `velocity`, `color`).
- A native **Unity VFX Graph 17 Binary 3D Vector Field (`.vf`, FourCC `VF_V`)** writer matching SideFX Labs' `Export Vector Field` schema.
- A procedural soft-radial sprite (`soft_radial_puff.png`) so all three effects render cleanly in Unity without external texture downloads.

In [ ]:
import json
import struct
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import numpy as np

SEED = 536
rng = np.random.default_rng(SEED)

EXPORT_DIR = Path('./lab13_vfx17_exports')
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# --- Multi-DCC Coordinate Conversions (Lecture 36, Slide 3) ---
def blender_rh_zup_to_unity_lh_yup(pts: np.ndarray) -> np.ndarray:
    """Convert (N, 3) points/vectors from Blender RH Z-up to Unity LH Y-up.
    Mapping: (x_b, y_b, z_b) -> (x_b, z_b, y_b), preserving meters (1 unit = 1 m).
    """
    pts = np.asarray(pts, dtype=np.float32)
    return np.stack([pts[:, 0], pts[:, 2], pts[:, 1]], axis=-1)

def houdini_rh_yup_to_unity_lh_yup(pts: np.ndarray) -> np.ndarray:
    """Convert (N, 3) points/vectors from Houdini RH Y-up to Unity LH Y-up.
    Mapping: negate Z (or X) to flip handedness: (x_h, y_h, z_h) -> (x_h, y_h, -z_h).
    """
    pts = np.asarray(pts, dtype=np.float32)
    return np.stack([pts[:, 0], pts[:, 1], -pts[:, 2]], axis=-1)

# --- Unity VFX Graph 17 Point Cache (.pcache) ASCII Exporter ---
def write_unity_pcache(filepath: Path, positions: np.ndarray, velocities: np.ndarray, colors: np.ndarray, comment: str) -> None:
    n = positions.shape[0]
    lines = [
        'pcache',
        'format ascii 1.0',
        f'comment {comment}',
        f'elements {n}',
        'property float position.x',
        'property float position.y',
        'property float position.z',
        'property float velocity.x',
        'property float velocity.y',
        'property float velocity.z',
        'property float color.r',
        'property float color.g',
        'property float color.b',
        'end_header',
    ]
    for i in range(n):
        px, py, pz = positions[i]
        vx, vy, vz = velocities[i]
        cr, cg, cb = colors[i]
        lines.append(f'{px:.6f} {py:.6f} {pz:.6f} {vx:.6f} {vy:.6f} {vz:.6f} {cr:.4f} {cg:.4f} {cb:.4f}')
    filepath.write_text('\n'.join(lines) + '\n', encoding='utf-8')

# --- Unity VFX Graph 17 3D Vector Field (.vf, VF_V) Binary Exporter ---
def write_unity_vf(filepath: Path, field_xyz: np.ndarray) -> None:
    """Write (nx, ny, nz, 3) float32 velocity grid to Unity VFX Graph .vf (VF_V) binary format.
    Unity expects 4-byte FourCC 'VF_V', three uint16 dims (nx, ny, nz), then float32 RGB
    ordered with X varying fastest, then Y, then Z.
    """
    nx, ny, nz, c = field_xyz.shape
    assert c == 3, 'Vector field must have 3 channels (vx, vy, vz).'
    # Transpose (nx, ny, nz, 3) -> (nz, ny, nx, 3) so C-order puts X fastest
    ordered = np.ascontiguousarray(np.transpose(field_xyz, (2, 1, 0, 3)), dtype='<f4')
    header = struct.pack('<4sHHH', b'VF_V', nx, ny, nz)
    with filepath.open('wb') as f:
        f.write(header)
        f.write(ordered.tobytes())

# --- Procedural Soft Radial Billboard Texture (for Fire, Smoke & Mist) ---
def generate_soft_radial_sprite(filepath: Path, size: int = 128) -> np.ndarray:
    coords = np.linspace(-1.0, 1.0, size, dtype=np.float32)
    xx, yy = np.meshgrid(coords, coords)
    r = np.sqrt(xx**2 + yy**2)
    alpha = np.clip(1.0 - r, 0.0, 1.0) ** 2.2
    rgba = np.ones((size, size, 4), dtype=np.float32)
    rgba[:, :, 3] = alpha
    mpimg.imsave(filepath, rgba)
    return rgba

sprite_path = EXPORT_DIR / 'soft_radial_puff.png'
generate_soft_radial_sprite(sprite_path)
print(f'Initialized export directory: {EXPORT_DIR.resolve()}')
print(f'Generated billboard sprite:   {sprite_path.name}')

---
## Block 1: Simplest Possible Fire Effect (Lagrangian Buoyancy + Curl Turbulence + Additive Quads)

**Lecture Alignment:** Lecture 35 Slide 2 (Lagrangian SoA lifecycle), Slide 5 (Unity VFX Graph `Initialize` / `Update` / `Output` contexts), and Slide 7 (*Case Study: Spark and Fire Simulation Pipeline*).

### Unity 6 / VFX Graph 17 Minimal Node Recipe (`SimpleFire.vfx`)
Create a new VFX Graph asset (*Create $\rightarrow$ Visual Effects $\rightarrow$ Visual Effect Graph $\rightarrow$ Minimal System*) and configure exactly four contexts:

| VFX Graph 17 Context | Minimal Blocks to Add | Exact Parameter Values |
| :--- | :--- | :--- |
| **1. Spawn** | `Constant Spawn Rate` | `Rate = 180` particles/sec |
| **2. Initialize Particle** | `Capacity`, `Set Position from Map` (or `Position (Circle)`), `Set Velocity Random`, `Set Lifetime Random` | `Capacity = 1024`; bind `fire_base_emitter.pcache` (`Position` + `Velocity`) OR `Circle Radius = 0.25 m`; `Lifetime = Random(0.45, 0.95) s` |
| **3. Update Particle** | `Force` (Buoyancy), `Turbulence` (Curl Noise), `Linear Drag` | `Force = (0, +3.6, 0) m/s^2`; `Turbulence Intensity = 1.4`, `Frequency = 1.8`, `Octaves = 2`; `Drag = 0.6` |
| **4. Output Particle Quad** | `Orient: Face Camera Plane`, `Set Size over Life`, `Set Color over Life` | `Main Texture = soft_radial_puff.png`; `Blend Mode = Additive`; `Size`: `0.22 -> 0.40 -> 0.05`; `HDR Color Gradient` ($t \in [0,1]$): White-Yellow `(4, 3.2, 1.5)` $\rightarrow$ Orange `(3, 0.9, 0.1)` $\rightarrow$ Dark Red `(0.4, 0.05, 0.0)` $\rightarrow$ Alpha `0` |

**Blender 4.2 Integration Note:** The cell below also writes `export_fire_emitter_blender4.py` so you can export an ember-bed emitter mesh from Blender 4.2 (Z-up) into Unity 6 (Y-up) via FBX or OpenUSD with applied transforms.

In [ ]:
# --- BLOCK 1: Fire .pcache Export, Blender 4.2 Helper Script & Lagrangian SoA Preview ---
N_FIRE_SEEDS = 512
theta = rng.uniform(0.0, 2.0 * np.pi, size=N_FIRE_SEEDS)
radius = 0.25 * np.sqrt(rng.uniform(0.0, 1.0, size=N_FIRE_SEEDS))

# Author in Unity LH Y-up meters (disc emitter on y = 0)
fire_pos = np.stack([radius * np.cos(theta), np.zeros(N_FIRE_SEEDS), radius * np.sin(theta)], axis=-1).astype(np.float32)
fire_vel = np.stack([
    rng.normal(0.0, 0.18, size=N_FIRE_SEEDS),
    rng.uniform(0.7, 1.5, size=N_FIRE_SEEDS),
    rng.normal(0.0, 0.18, size=N_FIRE_SEEDS),
], axis=-1).astype(np.float32)
# Core-to-rim heat color encoding
core_heat = np.clip(1.0 - (radius / 0.25), 0.0, 1.0)
fire_col = np.stack([np.ones(N_FIRE_SEEDS), 0.35 + 0.60 * core_heat, 0.05 + 0.45 * core_heat], axis=-1).astype(np.float32)

fire_pcache_path = EXPORT_DIR / 'fire_base_emitter.pcache'
write_unity_pcache(fire_pcache_path, fire_pos, fire_vel, fire_col, 'Block 1 Fire Disc Emitter (Unity LH Y-up)')

# Emit Blender 4.2 LTS bpy export script for a fire-pit ember emitter mesh (Lecture 36, Slide 3)
blender_fire_script = """import bpy

# Run inside Blender 4.2 LTS to export a clean Y-up emitter mesh for Unity 6 VFX Graph 17
bpy.ops.mesh.primitive_circle_add(vertices=32, radius=0.25, fill_type='NGON', location=(0.0, 0.0, 0.0))
emitter = bpy.context.active_object
emitter.name = 'FireEmberEmitter'
bpy.ops.object.transform_apply(location=True, rotation=True, scale=True)

# Export as FBX with explicit Unity Y-up / -Z forward axis conversion
bpy.ops.export_scene.fbx(
    filepath='fire_ember_emitter.fbx',
    use_selection=True,
    apply_scale_options='FBX_SCALE_UNITS',
    axis_forward='-Z',
    axis_up='Y',
)
"""
(EXPORT_DIR / 'export_fire_emitter_blender4.py').write_text(blender_fire_script, encoding='utf-8')

# Minimal SoA Lagrangian Fire Simulation Preview (Semi-Implicit Euler + Buoyancy + Turbulence)
dt = 1.0 / 60.0
n_p = 400
p_pos = fire_pos[:n_p].copy()
p_vel = fire_vel[:n_p].copy()
p_life = rng.uniform(0.45, 0.95, size=n_p)
p_age = rng.uniform(0.0, 0.95, size=n_p) * p_life  # steady-state staggered ages

for step in range(45):
    t_norm = np.clip(p_age / p_life, 0.0, 1.0)
    # Upward buoyancy + simple divergence-free 2D/3D sinusoidal curl turbulence
    buoyancy = np.array([0.0, 3.6, 0.0], dtype=np.float32)
    curl_x = 1.4 * np.sin(3.5 * p_pos[:, 1] + step * 0.12)
    curl_z = 1.4 * np.cos(3.5 * p_pos[:, 1] - step * 0.12)
    turb = np.stack([curl_x, np.zeros(n_p), curl_z], axis=-1)
    p_vel = (p_vel + (buoyancy + turb) * dt) * np.exp(-0.6 * dt)
    p_pos += p_vel * dt
    p_age += dt
    dead = p_age >= p_life
    if np.any(dead):
        k = int(np.sum(dead))
        idx = rng.integers(0, N_FIRE_SEEDS, size=k)
        p_pos[dead] = fire_pos[idx]
        p_vel[dead] = fire_vel[idx]
        p_age[dead] = 0.0

t_norm = np.clip(p_age / p_life, 0.0, 1.0)
sizes = 220.0 * (1.0 - 0.85 * t_norm) * (0.4 + 0.6 * np.sin(np.pi * t_norm))
colors_rgba = np.stack([
    np.clip(1.0 - 0.55 * t_norm, 0.0, 1.0),
    np.clip(0.95 - 1.15 * t_norm, 0.0, 1.0),
    np.clip(0.55 - 1.50 * t_norm, 0.0, 1.0),
    np.clip(0.85 * (1.0 - t_norm), 0.05, 0.85),
], axis=-1)

fig, ax = plt.subplots(figsize=(5.5, 5.0))
ax.set_facecolor('#111116')
ax.scatter(p_pos[:, 0], p_pos[:, 1], s=sizes, c=colors_rgba, edgecolors='none')
ax.set_xlim(-0.75, 0.75)
ax.set_ylim(-0.05, 2.20)
ax.set_xlabel('Unity X (m)')
ax.set_ylabel('Unity Y / Up (m)')
ax.set_title('Block 1 Preview: Simplest Fire (SoA Buoyancy + Curl Turbulence)', fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Exported: {fire_pcache_path} and export_fire_emitter_blender4.py')

---
## Block 2: Simplest Possible Smoke Plume Effect (Divergence-Free `.vf` Vector Field + Lit Alpha Billboards)

**Lecture Alignment:** Lecture 35 Slide 3 (Eulerian Navier–Stokes incompressibility $\nabla \cdot \mathbf{u} = 0$, *GPU Gems* Ch. 38) & Slide 6 (Billboard rendering, alpha sorting, Soft Particles) + Lecture 36 Slides 4–5 (Houdini $\rightarrow$ Unity 6 `.vf` 3D Vector Field bridge with `sRGB = Off`, `Compression = None`).

### Mathematical Construction of the 3D Vector Field (`smoke_plume_field.vf`)
To guarantee an **incompressible (divergence-free)** swirling smoke velocity field without an iterative Poisson solve in Python, we define a smooth 3D vector potential $\boldsymbol{\psi}(x, y, z)$ and take its curl plus a constant vertical thermal updraft $\mathbf{u}_0 = (0, +1.1, 0)$:
$$\mathbf{u}(x, y, z) = \nabla \times \boldsymbol{\psi}(x, y, z) + \mathbf{u}_0 \implies \nabla \cdot \mathbf{u} = \nabla \cdot (\nabla \times \boldsymbol{\psi}) + 0 = 0$$
In **Houdini**, the identical grid is authored in a Pyro/Volume SOP (`Volume VOP` curl noise + buoyancy) and exported via SideFX Labs **ROP Vector Field (`.vf`)**.

### Unity 6 / VFX Graph 17 Minimal Node Recipe (`SimpleSmoke.vfx`)
Import `smoke_plume_field.vf` into Unity 6 (verify `Texture Shape = 3D`, `sRGB = Off`, `Filter Mode = Bilinear`, `Wrap Mode = Clamp`).

| VFX Graph 17 Context | Minimal Blocks to Add | Exact Parameter Values |
| :--- | :--- | :--- |
| **1. Spawn** | `Constant Spawn Rate` | `Rate = 35` particles/sec *(keep low to bound alpha overdraw per Lecture 35 Slide 8)* |
| **2. Initialize Particle** | `Capacity`, `Position (Sphere)`, `Set Lifetime Random`, `Set Angle.Z Random` | `Capacity = 512`; `Sphere Radius = 0.22 m` centered at `(0, 0.8, 0)`; `Lifetime = Random(2.5, 4.2) s`; `Angle.Z = Random(0, 360) deg` |
| **3. Update Particle** | `Vector Field Force`, `Linear Drag`, `Angular Velocity` | Bind `smoke_plume_field.vf` (`Texture3D`), `Volume Center = (0, 2.0, 0)`, `Size = (2.4, 3.6, 2.4)`, `Intensity = 1.8`, `Drag = 1.2`; `Angular Velocity.Z = 18 deg/s` |
| **4. Output Particle Lit Quad** | `Orient: Face Camera Plane`, `Set Size over Life`, `Set Color & Alpha over Life` | `Blend Mode = Alpha`; enable **Soft Particles** (`Soft Particles Fade Distance = 0.6 m`); `Size`: `0.35 -> 1.45 m`; `Color`: warm mid-grey `(0.35, 0.35, 0.37)` fading alpha `0.0 -> 0.38 -> 0.0` |

In [ ]:
# --- BLOCK 2: Divergence-Free 3D Vector Field (.vf) Export & Smoke Plume Preview ---
NX, NY, NZ = 16, 24, 16
xs = np.linspace(-1.2, 1.2, NX, dtype=np.float32)
ys = np.linspace(0.0, 3.6, NY, dtype=np.float32)
zs = np.linspace(-1.2, 1.2, NZ, dtype=np.float32)
dx, dy, dz = float(xs[1] - xs[0]), float(ys[1] - ys[0]), float(zs[1] - zs[0])

X, Y, Z = np.meshgrid(xs, ys, zs, indexing='ij')

# Smooth vector potential psi = (psi_x, psi_y, psi_z) creating buoyant vortex eddies
envelope = np.exp(-0.45 * (X**2 + Z**2))
psi_x = 0.28 * envelope * np.sin(1.8 * Y + 1.2 * Z)
psi_y = 0.35 * envelope * np.cos(1.4 * X + 1.4 * Z)
psi_z = 0.28 * envelope * np.cos(1.8 * Y - 1.2 * X)

# Discrete curl u = curl(psi) + constant thermal buoyancy (0, 1.15, 0)
d_psiz_dy = np.gradient(psi_z, dy, axis=1)
d_psiy_dz = np.gradient(psi_y, dz, axis=2)
d_psix_dz = np.gradient(psi_x, dz, axis=2)
d_psiz_dx = np.gradient(psi_z, dx, axis=0)
d_psiy_dx = np.gradient(psi_y, dx, axis=0)
d_psix_dy = np.gradient(psi_x, dy, axis=1)

ux = d_psiz_dy - d_psiy_dz
uy = (d_psix_dz - d_psiz_dx) + 1.15
uz = d_psiy_dx - d_psix_dy

# Verify interior incompressibility (Lecture 35 Slide 3: div(u) == 0)
div_u = np.gradient(ux, dx, axis=0) + np.gradient(uy, dy, axis=1) + np.gradient(uz, dz, axis=2)
interior_div_rms = float(np.sqrt(np.mean(div_u[2:-2, 2:-2, 2:-2] ** 2)))

smoke_vf = np.stack([ux, uy, uz], axis=-1).astype(np.float32)
smoke_vf_path = EXPORT_DIR / 'smoke_plume_field.vf'
write_unity_vf(smoke_vf_path, smoke_vf)

# Advect Lagrangian smoke particles through the 3D vector field
def sample_vf_nearest(pos_arr: np.ndarray) -> np.ndarray:
    ix = np.clip(((pos_arr[:, 0] - xs[0]) / dx).astype(int), 0, NX - 1)
    iy = np.clip(((pos_arr[:, 1] - ys[0]) / dy).astype(int), 0, NY - 1)
    iz = np.clip(((pos_arr[:, 2] - zs[0]) / dz).astype(int), 0, NZ - 1)
    return smoke_vf[ix, iy, iz]

n_s = 220
s_pos = np.stack([rng.normal(0.0, 0.14, n_s), rng.uniform(0.15, 0.35, n_s), rng.normal(0.0, 0.14, n_s)], axis=-1).astype(np.float32)
s_vel = np.zeros((n_s, 3), dtype=np.float32)
s_life = rng.uniform(2.5, 4.0, n_s)
s_age = rng.uniform(0.0, 1.0, n_s) * s_life

for _ in range(90):
    v_target = sample_vf_nearest(s_pos)
    # Vector Field Force with drag relaxation toward grid velocity
    s_vel += 1.8 * (v_target - s_vel) * dt
    s_pos += s_vel * dt
    s_age += dt
    dead = s_age >= s_life
    if np.any(dead):
        k = int(np.sum(dead))
        s_pos[dead] = np.stack([rng.normal(0.0, 0.14, k), rng.uniform(0.15, 0.35, k), rng.normal(0.0, 0.14, k)], axis=-1)
        s_vel[dead] = 0.0
        s_age[dead] = 0.0

s_norm = np.clip(s_age / s_life, 0.0, 1.0)
s_sizes = 90.0 + 480.0 * s_norm
s_alpha = 0.35 * np.sin(np.pi * s_norm)
s_colors = np.stack([np.full(n_s, 0.62), np.full(n_s, 0.64), np.full(n_s, 0.68), s_alpha], axis=-1)

fig, (ax_vf, ax_sm) = plt.subplots(1, 2, figsize=(10.5, 4.8))
mid_z = NZ // 2
ax_vf.quiver(X[:, :, mid_z], Y[:, :, mid_z], ux[:, :, mid_z], uy[:, :, mid_z], color='#0072B2', alpha=0.85)
ax_vf.set_title(f'3D Vector Field Slice (z=0) | RMS |div(u)| = {interior_div_rms:.2e}', fontweight='bold', fontsize=10)
ax_vf.set_xlabel('Unity X (m)')
ax_vf.set_ylabel('Unity Y / Up (m)')

ax_sm.set_facecolor('#1a1d24')
ax_sm.scatter(s_pos[:, 0], s_pos[:, 1], s=s_sizes, c=s_colors, edgecolors='none')
ax_sm.set_xlim(-1.2, 1.2)
ax_sm.set_ylim(0.0, 3.6)
ax_sm.set_xlabel('Unity X (m)')
ax_sm.set_ylabel('Unity Y / Up (m)')
ax_sm.set_title('Block 2 Preview: Smoke Plume Advected by .vf Grid', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.show()
print(f'Exported binary Unity VFX 17 Vector Field: {smoke_vf_path} ({smoke_vf_path.stat().st_size} bytes)')

---
## Block 3: Simplest Possible Waterfall Effect (Multi-DCC Lip `.pcache` + Gravity + Analytical SDF Plane Collision)

**Lecture Alignment:** Lecture 35 Slides 2, 4, and 7 (fixed-step semi-implicit Euler integration, deterministic gravity, velocity-stretched quads, and implicit SDF plane collision) + Lecture 36 Slide 3 (`Blender 4.2 Z-up -> Unity 6 LH Y-up` emitter coordinate validation).

### Unity 6 / VFX Graph 17 Minimal Node Recipe (`SimpleWaterfall.vfx`)
Build two lightweight strips/systems inside a single VFX Graph sharing the basin floor plane ($y = 0$):

| System / Context | Minimal Blocks to Add | Exact Parameter Values |
| :--- | :--- | :--- |
| **System A — Falling Water Sheet** | | |
| `Spawn` | `Constant Spawn Rate` | `Rate = 600` particles/sec |
| `Initialize` | `Capacity`, `Set Position from Map` (`waterfall_lip.pcache`), `Set Velocity from Map`, `Set Lifetime Random` | `Capacity = 2048`; cliff lip at `y = 4.0 m`, outward velocity `vz = +1.4 m/s`; `Lifetime = Random(1.0, 1.35) s` |
| `Update` | `Gravity`, `Collide with Plane` (Analytical SDF $d(\mathbf{p}) = p_y$) | `Gravity = (0, -9.81, 0) m/s^2`; `Plane Position = (0, 0, 0)`, `Normal = (0, 1, 0)`, `Bounce = 0.22`, `Roughness = 0.35`, `Lifetime Loss = 0.55` |
| `Output Particle Quad` | `Orient: Along Velocity`, `Set Scale (X, Y)`, `Set Color over Life` | `Blend Mode = Alpha`; `Scale.X = 0.09`, `Scale.Y = 0.45` (stretched water streak); `Color`: translucent cyan-white `(0.75, 0.90, 0.98)` fading alpha `0.65 -> 0.15` |
| **System B — Basin Splash Mist (Optional Minimal Layer)** | | |
| `Spawn` $\rightarrow$ `Output` | `Constant Spawn Rate (45/s)` along basin impact band `(z = 1.25 m, y = 0.08 m)` | Upward velocity `Random(0.4, 1.1) m/s`, `Drag = 1.5`, `Output Particle Lit Quad` (`Soft Particles = On`, `Size 0.4 -> 1.3 m`, `Alpha <= 0.25`) |

In [ ]:
# --- BLOCK 3: Waterfall Cliff-Lip .pcache (Blender Z-up -> Unity LH Y-up) & SDF Collision Preview ---
N_LIP = 512

# 1. Author cliff lip in Blender 4.2 coordinates (RH Z-up: height is Z = 4.0 m, forward spill is +Y)
lip_blender_pos = np.stack([
    rng.uniform(-1.0, 1.0, size=N_LIP),   # width along X
    rng.uniform(-0.08, 0.08, size=N_LIP), # depth along Blender Y
    np.full(N_LIP, 4.0),                  # height along Blender Z = 4.0 m
], axis=-1).astype(np.float32)

lip_blender_vel = np.stack([
    rng.normal(0.0, 0.08, size=N_LIP),
    rng.uniform(1.15, 1.65, size=N_LIP),  # outward spill along Blender +Y
    rng.uniform(-0.25, 0.05, size=N_LIP), # slight drop along Blender Z
], axis=-1).astype(np.float32)

# 2. Apply explicit coordinate transform to Unity 6 LH Y-up (Lecture 36, Slide 3)
lip_unity_pos = blender_rh_zup_to_unity_lh_yup(lip_blender_pos)
lip_unity_vel = blender_rh_zup_to_unity_lh_yup(lip_blender_vel)
lip_unity_col = np.tile(np.array([0.75, 0.90, 0.98], dtype=np.float32), (N_LIP, 1))

waterfall_pcache_path = EXPORT_DIR / 'waterfall_lip.pcache'
write_unity_pcache(waterfall_pcache_path, lip_unity_pos, lip_unity_vel, lip_unity_col,
                   'Block 3 Waterfall Lip Emitter (Converted from Blender Z-up to Unity LH Y-up)')

# 3. Semi-Implicit Euler Simulation + Analytical Plane SDF Collision (d(p) = y = 0)
n_w = 500
w_pos = lip_unity_pos[:n_w].copy()
w_vel = lip_unity_vel[:n_w].copy()
w_life = rng.uniform(1.05, 1.40, size=n_w)
w_age = rng.uniform(0.0, 1.0, size=n_w) * w_life
w_mist = np.zeros(n_w, dtype=bool)
gravity = np.array([0.0, -9.81, 0.0], dtype=np.float32)

for _ in range(95):
    # Semi-implicit Euler update (Lecture 35, Slide 4)
    w_vel[~w_mist] += gravity * dt
    w_vel[w_mist] = (w_vel[w_mist] + np.array([0.0, 0.9, 0.0]) * dt) * np.exp(-1.5 * dt)
    w_pos += w_vel * dt
    w_age += dt

    # Analytical Plane SDF collision at y <= 0.0
    hit = (w_pos[:, 1] <= 0.0) & (~w_mist)
    if np.any(hit):
        k = int(np.sum(hit))
        w_pos[hit, 1] = 0.02
        w_vel[hit, 0] += rng.normal(0.0, 0.65, size=k)
        w_vel[hit, 1] = np.abs(w_vel[hit, 1]) * 0.22 + rng.uniform(0.2, 0.7, size=k)
        w_vel[hit, 2] = w_vel[hit, 2] * 0.35 + rng.normal(0.0, 0.55, size=k)
        w_age[hit] += 0.45 * w_life[hit]  # Lifetime Loss on impact
        w_mist[hit] = True

    dead = w_age >= w_life
    if np.any(dead):
        k = int(np.sum(dead))
        idx = rng.integers(0, N_LIP, size=k)
        w_pos[dead] = lip_unity_pos[idx]
        w_vel[dead] = lip_unity_vel[idx]
        w_age[dead] = 0.0
        w_mist[dead] = False

# Side-profile plot (Unity Z forward vs Unity Y up)
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.set_facecolor('#0f1a24')
ax.axhline(0.0, color='#56B4E9', linewidth=2.2, label='Basin Water Surface (SDF Plane y = 0)')
ax.plot([-0.4, 0.0, 0.0], [4.0, 4.0, 0.0], color='#8c7b6b', linewidth=3.0, label='Cliff Geometry (Blender -> Unity)')
ax.scatter(w_pos[~w_mist, 2], w_pos[~w_mist, 1], s=18, c='#56B4E9', alpha=0.72, label='Falling Water Sheet (Stretched Quads)')
ax.scatter(w_pos[w_mist, 2], w_pos[w_mist, 1], s=140, c='#e6f4ff', alpha=0.25, edgecolors='none', label='Basin Splash & Mist (Soft Alpha)')
ax.set_xlim(-0.4, 2.8)
ax.set_ylim(-0.2, 4.35)
ax.set_xlabel('Unity Z / Spill Forward (m)')
ax.set_ylabel('Unity Y / Height (m)')
ax.set_title('Block 3 Preview: Simplest Waterfall Side Profile (Gravity + SDF Plane Collision)', fontweight='bold')
ax.legend(frameon=False, loc='upper right', fontsize=8.5, labelcolor='white')
plt.tight_layout()
plt.show()
print(f'Exported transformed Waterfall Lip Point Cache: {waterfall_pcache_path}')

---
## Block 4: Unity 6 C# Profiler Harness & Healy Ch. 5–6 Quantitative Benchmark Analysis

**Lecture Alignment:** Lecture 35 Slide 8 (*Performance Optimization & Bottlenecks*: fill-rate overdraw, LOD scaling, particle budgets) + Lecture 36 Slides 6–7 (*Quantitative Profiling & Visualizing Benchmarks with 95% CIs*).

Run the cell below to:
1. Emit `VfxLab13Profiler.cs`, a drop-in Unity 6 C# script using `UnityEngine.VFX.VisualEffect` and `Unity.Profiling.ProfilerRecorder` to log per-frame CPU/GPU frame time and `aliveParticleCount` to CSV.
2. Generate a **Healy Ch. 5–6 small-multiple benchmark chart** (Okabe–Ito colorblind-safe palette, zero top/right spines, 95% confidence intervals) comparing Fire, Smoke, and Waterfall scaling across particle budgets ($10^3, 10^4, 10^5$)—which you will replace with your real Unity Profiler CSV capture for Phase 3.

In [ ]:
# --- BLOCK 4: Unity 6 VFX 17 C# Benchmark Harness & Healy Ch. 5-6 Scaling Plot ---
unity_cs_harness = """using System.Collections.Generic;
using System.IO;
using Unity.Profiling;
using UnityEngine;
using UnityEngine.VFX;

public class VfxLab13Profiler : MonoBehaviour
{
    [Header("Assign Fire, Smoke, or Waterfall VisualEffect Component")]
    public VisualEffect targetVfx;
    public string effectLabel = "Fire";
    public int framesToCapture = 300;

    private ProfilerRecorder _mainThreadRecorder;
    private readonly List<string> _rows = new List<string> { "effect,frame,alive_particles,frame_time_ms" };
    private int _frameCount = 0;

    void OnEnable()
    {
        _mainThreadRecorder = ProfilerRecorder.StartNew(ProfilerCategory.Internal, "Main Thread", 15);
    }

    void OnDisable()
    {
        _mainThreadRecorder.Dispose();
    }

    void Update()
    {
        if (targetVfx == null || _frameCount >= framesToCapture) return;
        float frameMs = _mainThreadRecorder.LastValue * 1e-6f;
        int alive = targetVfx.aliveParticleCount;
        _rows.Add($"{effectLabel},{_frameCount},{alive},{frameMs:F4}");
        _frameCount++;

        if (_frameCount == framesToCapture)
        {
            string outPath = Path.Combine(Application.persistentDataPath, $"vfx17_bench_{effectLabel}.csv");
            File.WriteAllLines(outPath, _rows);
            Debug.Log($"[Lab 13] Saved {framesToCapture} frames to {outPath}");
        }
    }
}
"""
cs_path = EXPORT_DIR / 'VfxLab13Profiler.cs'
cs_path.write_text(unity_cs_harness, encoding='utf-8')

# Reference Healy Ch. 5-6 Small-Multiple Scaling Plot (Replace with your Unity CSV logs in Phase 3)
budgets = np.array([1000, 10000, 100000])
effects = [
    ('Fire (Additive Small Quads)', '#E69F00', [1.8, 3.1, 8.9], [0.15, 0.28, 0.72]),
    ('Smoke (Lit Soft Alpha Overdraw)', '#0072B2', [2.6, 6.8, 22.4], [0.22, 0.55, 1.85]),
    ('Waterfall (Stretched + SDF Plane)', '#009E73', [2.1, 4.4, 12.6], [0.18, 0.36, 0.95]),
]

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0), sharey=True)
for ax, (title, color, means, cis) in zip(axes, effects):
    ax.errorbar(budgets, means, yerr=cis, fmt='o-', color=color, linewidth=2.2, markersize=7, capsize=4)
    ax.axhline(16.67, color='#D55E00', linestyle='--', linewidth=1.4, label='60 FPS Budget (16.67 ms)')
    for x_b, y_m in zip(budgets, means):
        ax.text(x_b, y_m + 1.1, f'{y_m:.1f} ms', ha='center', fontsize=8.5, fontweight='bold', color=color)
    ax.set_xscale('log')
    ax.set_xticks(budgets)
    ax.set_xticklabels(['1K', '10K', '100K'])
    ax.set_xlabel('Active Particle Budget (log scale)')
    ax.set_title(title, loc='left', fontweight='bold', fontsize=10)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', linestyle=':', alpha=0.35)
    ax.legend(frameon=False, loc='upper left', fontsize=8)

axes[0].set_ylabel('Frame Time ms (Mean +/- 95% CI)')
fig.suptitle('Reference Scaling Profile: Overdraw & Compute Cost Across VFX Graph 17 Effects (Healy Ch. 5-6)', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

manifest = {
    'export_dir': str(EXPORT_DIR),
    'generated_artifacts': sorted([p.name for p in EXPORT_DIR.iterdir()]),
}
print(json.dumps(manifest, indent=2))

---
## Week 13 Lab Verification & Import Checklist (Unity 6 + VFX Graph 17)

- [ ] **Package Check:** Verify `com.unity.visualeffectgraph` (`17.x`) is installed via Unity 6 Package Manager alongside URP 17 or HDRP 17.
- [ ] **Block 1 (Fire):** Import `fire_base_emitter.pcache` and `soft_radial_puff.png`; verify additive blend mode, upward buoyancy, curl turbulence, and white $\rightarrow$ orange $\rightarrow$ dark-red lifetime gradient.
- [ ] **Block 2 (Smoke):** Import `smoke_plume_field.vf`; verify `Texture3D` import settings (`sRGB = Off`, `Filter Mode = Bilinear`, `Wrap Mode = Clamp`) and enable **Soft Particles** on the Lit Quad output to avoid hard floor-intersection seams.
- [ ] **Block 3 (Waterfall):** Import `waterfall_lip.pcache`; confirm the Blender Z-up $\rightarrow$ Unity Y-up transform places the lip at `y = 4.0 m` spilling forward along `+Z`, and verify analytical plane collision at `y = 0`.
- [ ] **Block 4 (Profiling):** Attach `VfxLab13Profiler.cs`, capture 300 frames for each effect, and replace the reference data in Block 4 with your measured CSV telemetry before committing clean interchange files (`.pcache`, `.vf`, `.cs`, `.ipynb`) to GitHub.